# 01 Coverage check

Is the energy-only sample large enough, or does the universe need widening? This notebook
builds the firm-month skeleton from cached pulls (it connects to WRDS only if one is
missing) and reports, per year, the firms in the matched panel and the rating downgrades and
defaults that have option coverage around them.

**Coverage level.** `LINK_THROUGH` sets how far linking goes:
`"surface"` (volatility-surface data in the month) needs the `optionm` library, `"secid"`
needs the OptionMetrics-CRSP link, and `"permno"` needs only CRSP. With `"permno"`, "matched"
means *rated and trading on CRSP*, which is an **upper bound** on option coverage.

In [ ]:
import dataclasses
import logging

import matplotlib.pyplot as plt
import pandas as pd

from src import checks, panel, ratings
from src.config import load_config
from src.wrds_conn import WrdsSession

LINK_THROUGH = "permno"  # "surface" once the account can read optionm

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
pd.set_option("display.width", 200, "display.max_columns", 30)

cfg = load_config()
with WrdsSession() as session:  # connects only for pulls that are not cached yet
    df = panel.firm_months(session, cfg, link_through=LINK_THROUGH)
    hist_gics = ratings.load_historical_gics(session, cfg)

sector = cfg.universe.gics_sector
energy = df["in_universe"]
all_ex_fin = ratings.universe_mask(df, dataclasses.replace(cfg.universe, gics_sector=None))
matched_on = checks.matched_column(df)
print(f"{len(df):,} rated firm-months, {df['gvkey'].nunique():,} firms; matched on {matched_on}")
print(f"universe: {cfg.universe}")

## 1. Ratings in the configured universe

In [ ]:
u = df[energy]
print(f"GICS {sector}: {u['gvkey'].nunique()} rated firms, {len(u):,} firm-months, "
      f"{u['datadate'].min():%Y-%m} to {u['datadate'].max():%Y-%m}")
(u["splticrm"].value_counts().reindex(list(ratings.RATING_SCALE)).dropna()
 .astype(int).rename("firm-months").to_frame().T)

## 2. Sector filter diagnostics

The sector comes from `comp.company`, which holds each firm's *current* GICS codes.
`comp.co_hgic` holds the codes in force at each date (from June 1999, when GICS launched),
so comparing the two shows how often "current" misassigns a firm-month.

In [ ]:
firms = df.drop_duplicates("gvkey")
print("rated firms by current GICS sector:")
print(firms["gsector"].value_counts(dropna=False).sort_index().to_dict())

h = ratings.add_historical_gics(df, hist_gics)
h = h[h["gsector_hist"].notna()]  # post-1999 months with a known historical sector
now, then = h["gsector"].eq(sector).fillna(False), h["gsector_hist"].eq(sector)
print(f"\nfirm-months since 1999-06 with a historical sector: {len(h):,}")
print(f"  sector {sector} now and then: {int((now & then).sum()):,}")
print(f"  sector {sector} now only:     {int((now & ~then).sum()):,} firm-months, "
      f"{h.loc[now & ~then, 'gvkey'].nunique()} firms (included, but were not {sector} at the time)")
print(f"  sector {sector} then only:    {int((~now & then).sum()):,} firm-months, "
      f"{h.loc[~now & then, 'gvkey'].nunique()} firms (excluded, but were {sector} at the time)")

In [ ]:
def by_firm(mask, sector_col):
    cols = ["gvkey", "conm", sector_col]
    g = h[mask].groupby(cols, dropna=False)
    return g.agg(firm_months=("datadate", "size"), defaults=("is_default_event", "sum")).sort_values(
        "firm_months", ascending=False)

print(f"Excluded by current GICS, but sector {sector} at the time:")
display(by_firm(~now & then, "gsector"))
print(f"Included by current GICS, but another sector at the time (top 15):")
display(by_firm(now & ~then, "gsector_hist").head(15))

## 3. Linking funnel, configured universe

In [ ]:
d = df[energy].assign(year=df["datadate"].dt.year)
steps = {"rated": d, "CRSP permno": d[d["has_permno"]]}
if "has_secid" in d:
    steps["OptionMetrics secid"] = d[d["has_secid"]]
if "has_options" in d:
    steps["surface data"] = d[d["has_options"]]
funnel = pd.DataFrame({name: s.groupby("year")["gvkey"].nunique() for name, s in steps.items()}).fillna(0).astype(int)
funnel.loc["all years"] = [s["gvkey"].nunique() for s in steps.values()]
funnel

## 4. Coverage summary

`matched_firms` are distinct firms per year in the matched panel. An event counts as
*with options* if the firm was matched in the month before it (up/downgrades) or in any of
the 12 months before it (defaults). Downgrades exclude moves into default, which are counted
as defaults. Events come from month-end ratings, so a rating that lasts under a month (such
as a brief SD around a distressed exchange) is not seen.

In [ ]:
cov_energy = checks.coverage_summary(df, energy)
cov_all = checks.coverage_summary(df, all_ex_fin)
print(f"GICS sector {sector}, matched on {cov_energy.attrs['matched_on']}")
cov_energy

In [ ]:
print(f"All sectors excluding financials, matched on {cov_all.attrs['matched_on']}")
cov_all

In [ ]:
key = ["matched_firms", "downgrades_w_options", "defaults_w_options_12m"]
side_by_side = pd.concat({f"GICS {sector}": cov_energy[key], "all ex-financials": cov_all[key]}, axis=1)
side_by_side["share of downgrades"] = (side_by_side[(f"GICS {sector}", "downgrades_w_options")]
                                       / side_by_side[("all ex-financials", "downgrades_w_options")]).round(3)
side_by_side

In [ ]:
# Reference palette slots 1-2 (blue, orange); ink and grid from the same reference.
BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#898781", "#e1e0d9", "#fcfcfb"

end = cfg.sample.ratings_end
last_full_year = end.year if end.month == 12 else end.year - 1  # a partial year would read as a cliff
yearly = cov_energy.drop("all years")
yearly.index = yearly.index.astype(int)
yearly = yearly[yearly.index <= last_full_year]
fig, axes = plt.subplots(1, 2, figsize=(12, 4), facecolor=SURFACE)
panels = [
    ("Firms per year", [("rated_firms", "rated", MUTED), ("matched_firms", "matched", BLUE)]),
    ("Events with option coverage", [("downgrades_w_options", "downgrades", BLUE),
                                     ("defaults_w_options_12m", "defaults", ORANGE)]),
]
for ax, (title, series) in zip(axes, panels):
    ax.set_facecolor(SURFACE)
    for col, label, color in series:
        ax.plot(yearly.index, yearly[col], color=color, lw=2, marker="o", ms=4, label=label)
        ax.annotate(label, (yearly.index[-1], yearly[col].iloc[-1]), xytext=(6, -3),
                    textcoords="offset points", color=INK, fontsize=9)
    ax.set_title(f"GICS {sector}: {title.lower()}", loc="left", color=INK, fontsize=11)
    ax.set_ylim(bottom=0)
    ax.set_xticks(range(yearly.index[0], yearly.index[-1] + 1, 4))
    ax.grid(axis="y", color=GRID, lw=0.8)
    ax.tick_params(colors=MUTED, labelcolor=INK, labelsize=9)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(MUTED)
    ax.legend(frameon=False, fontsize=9, labelcolor=INK, loc="upper left")
fig.text(0.01, -0.02, f"Matched on {cov_energy.attrs['matched_on']}. Full calendar years only; "
         f"ratings end {end:%B %Y}.", color=MUTED, fontsize=8)
plt.tight_layout()

## 5. Open decisions: MLP/LLC units and coal

Flags only; with `include_mlps` / `include_coal` set to null these firms stay in the sample.
Units are CRSP share codes 70-79 in the month; coal is GICS sub-industry 10102050 (current).

In [ ]:
groups = {f"all GICS {sector}": energy, "MLP/LLC units": energy & df["is_unit"], "coal": energy & df["is_coal"]}
cols = ["matched_firms", "matched_firm_months", "downgrades_w_options", "defaults_w_options_12m"]
pd.DataFrame({name: checks.coverage_summary(df, mask).loc["all years", cols] for name, mask in groups.items()}).T

## 6. Known case: Chesapeake Energy

In [ ]:
checks.check_chesapeake(df)  # raises if the 2014-2016 deterioration is missing

## 7. Firm-count jumps and cliffs

In [ ]:
counts = checks.check_firm_counts(df[energy & df[matched_on]], "matched, configured universe")
print(f"{int(counts['flag'].sum())} flagged months")
counts[counts["flag"]]